In [2]:
!pip install pyspark
from pyspark import SparkConf, SparkContext
conf = SparkConf().setAppName("Q3").setMaster("local[*]")
sc = SparkContext.getOrCreate(conf)


In [4]:
import os, urllib.request
os.makedirs("data/World Hapiness Report", exist_ok=True)
base = "https://raw.githubusercontent.com/nongaussian/class-2024-datamining/main/data/World%20Hapiness%20Report/"
for y in [2015, 2016, 2017, 2018, 2019]:
    urllib.request.urlretrieve(f"{base}{y}.csv", f"data/World Hapiness Report/{y}.csv")
    print(f"downloaded {y}.csv")


downloaded 2015.csv
downloaded 2016.csv
downloaded 2017.csv
downloaded 2018.csv
downloaded 2019.csv


In [5]:
from functools import reduce

years = [2015, 2016, 2017, 2018, 2019]
data_dir = "data/World Hapiness Report"

# 연도별로 표기 방식이 다른 일부 국가명을 통일
name_map = {
    "Taiwan Province of China": "Taiwan",
    "Trinidad & Tobago": "Trinidad and Tobago",
    "North Cyprus": "Northern Cyprus"
}

# 각 파일의 첫 번째 파티션에서만 헤더(첫 줄)를 건너뛰기
def skip_header(partition_id, iterator):
    iterator = iter(iterator)
    if partition_id == 0:
        next(iterator, None)
    return iterator

# CSV 한 줄을 ((국가, 연도), 행복 점수) 형태로 파싱
def parse_line(line, year):
    parts = line.split(",")
    if len(parts) < 3:
        return None
    country = parts[0].strip()
    country = name_map.get(country, country)
    score = float(parts[2])
    return ((country, year), score)

# 5개 연도 데이터를 각각 RDD로 읽기
year_rdds = []
for year in years:
    path = f"{data_dir}/{year}.csv"
    lines = sc.textFile(path)
    rdd = (
        lines
        .mapPartitionsWithIndex(skip_header)
        .map(lambda line, y=year: parse_line(line, y))
        .filter(lambda x: x is not None)
    )
    year_rdds.append(rdd)

# 5개 RDD를 하나로 합치기
all_data = reduce(
    lambda rdd1, rdd2: rdd1.union(rdd2),
    year_rdds
)

# (국가, (연도, 행복 점수)) 형태로 변환 후 국가별로 묶기
by_country = (
    all_data
    .map(lambda x: (x[0][0], (x[0][1], x[1])))
    .groupByKey()
)

# 한 국가의 5년 행복 점수가 매년 전년도 이상인지 판단
def is_valid_country(record):
    country, values = record
    scores = dict(values)
    if set(scores.keys()) != set(years):
        return False
    return (
        scores[2016] >= scores[2015]
        and scores[2017] >= scores[2016]
        and scores[2018] >= scores[2017]
        and scores[2019] >= scores[2018]
    )

# 조건을 만족하는 국가만 남기고 이름순 정렬
result = (
    by_country
    .filter(is_valid_country)
    .keys()
    .sortBy(lambda country: country)
)

# 마지막 action
for country in result.collect():
    print(country)


Benin
Burkina Faso
Cambodia
Cameroon
Chad
Congo (Brazzaville)
Czech Republic
Dominican Republic
Estonia
Finland
Gabon
Honduras
Hungary
Ivory Coast
Latvia
Malta
Mongolia
Niger
Philippines
Poland
Portugal
Romania
Senegal
Serbia
Slovakia
Syria
Taiwan
Tajikistan
Togo
Trinidad and Tobago
